### 1. Load & Convert PDF to Text

In [3]:
from pypdf import PdfReader
import os

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

text = ""

file_path = "data.txt"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
else:
    folder_path = "data"
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            pdf_path = os.path.join(folder_path, file)
            text += extract_text_from_pdf(pdf_path)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

print(text[:1000])

1 
 
 
 
 
 
 
 
 
 
 
 
The Bharatiya Nyaya Sanhita, 2023 
 (ACT NO. 45 OF 2023) 
[As on the 6th October, 2025] 
  
2 
 
LIST OF ABBREVIATIONS USED 
G.S.R.              . . . .            .            for           General Statutory Rules.  
S.O.                 . . . . .            ,,              Statutory Order. 
Notifn.  . . . . .  ,,    Notification. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
THE BHARATIYA NYAYA SANHITA, 2023 
________ 
ARRANGEMENT OF SECTIONS 
________ 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1. Short title, commencement and application. 
2. Definitions. 
3. General explanations. 
CHAPTER II 
OF PUNISHMENTS 
4. Punishments. 
5. Commutation of sentence. 
6. Fractions of terms of punishment. 
7. Sentence may be (in certain cases of imprisonment) wholly or partly rigorous or simple. 
8. Amount of fine, liability in default of payment of fine, etc. 
9. Limit of punishment of offence made up of several offences. 
10. Punishment of person guilt

### 2. Convert Text → Dataset

In [4]:
from datasets import Dataset

def split_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = split_text(text)

dataset = Dataset.from_dict({"text": chunks})

### 3. Load Model + Tokenizer

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = 'distilgpt2' # lightweight

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map='auto') # uses gpu

### 4. Apply LoRA

In [6]:
from peft import LoraConfig, get_peft_model


In [8]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r = 4,
    lora_alpha = 16,
    target_modules = ['c_attn'],
    lora_dropout=0.1,
    bias='none',
    task_type = 'CAUSAL_LLM'
)

model = get_peft_model(model,lora_config)

W0329 00:12:21.045000 45812 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\peft\tuners\lora\layer.py:1119: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


### 5. Tokenization

In [9]:
def tokenize_function(example):
    return tokenizer(
        example['text'],
        padding='max_length',
        truncation = True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 8592/8592 [00:04<00:00, 1954.10 examples/s]


### 6. Training

In [13]:
tokenized_dataset = tokenized_dataset.map(
    lambda x: {
        "input_ids": x["input_ids"],
        "attention_mask": x["attention_mask"],
        "labels": x["input_ids"]
    },
    remove_columns=tokenized_dataset.column_names
)






































































































Map: 100%|██████████| 8592/8592 [00:14<00:00, 609.79 examples/s]


In [14]:
model.save_pretrained("lora-law-model")
tokenizer.save_pretrained("lora-law-model")

c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('lora-law-model\\tokenizer_config.json',
 'lora-law-model\\special_tokens_map.json',
 'lora-law-model\\vocab.json',
 'lora-law-model\\merges.txt',
 'lora-law-model\\added_tokens.json',
 'lora-law-model\\tokenizer.json')

In [16]:
model = model.merge_and_unload()   # 🔥 FIX

from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = "What is Indian Penal Code?"

print(pipe(prompt, max_length=200)[0]["generated_text"])

What is Indian Penal Code? This article is not meant to provide legal advice, to provide any guidance or advice for Indian law enforcement. Please stop by subscribing to the Law Enforcement Lawyer's Manual. Read the Law Enforcement Manual from the Information Centre's website and look at the legal advice here for more information. The legal knowledge of Indian criminal and civil law is not available to you. In the same article, some details of Indian legal practices, including laws such as criminal and civil law, is not available to you.




Indian Penal Code (IV)
A person, usually under age 18 who has to enter in the United States who has committed a crime or to commit a crime that it is a crime of a crime of a murder committed to or from any other country who has been abroad. Indian Penal Code for the general and non-criminal means only if the person has a person of any age or nationality at which such person is in full force or in danger of arrest


### 7. QloRA

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "microsoft/phi-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [21]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

`low_cpu_mem_usage` was None, now set to True since model is quantized.


: 

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # 🔥 for phi-2
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qlora-law",
    per_device_train_batch_size=1, # batch size
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = "Question: What is Indian Penal Code?\nAnswer:"

print(pipe(prompt, max_length=200)[0]["generated_text"])

### 10.complete code 1

In [7]:
# ============================================================
# ⚖️  AI Vakeel — QLoRA Fine-Tuning on Legal PDFs
# Model  : microsoft/phi-2
# Method : QLoRA (4-bit)
# GPU    : GTX 1650 Ti (4 GB VRAM)
# ============================================================
# Before running, install the Windows bitsandbytes wheel:
#
#   pip install https://github.com/jllllll/bitsandbytes-windows-webui/releases/download/wheels/bitsandbytes-0.41.1-py3-none-win_amd64.whl --no-deps --force-reinstall
#
# Required versions:
#   torch          2.1.0+cu118
#   transformers   4.36.2
#   tokenizers     0.15.2
#   peft           0.7.1
#   bitsandbytes   0.41.1
#   accelerate     0.25.0
# ============================================================


# ============================================================
# SECTION 1 — Sanity Check
# ============================================================
import torch
import transformers, peft, bitsandbytes, accelerate, tokenizers

print("=" * 50)
print("        ENVIRONMENT SANITY CHECK")
print("=" * 50)
print(f"torch:          {torch.__version__}")         # ✅ 2.1.0+cu118
print(f"CUDA available: {torch.cuda.is_available()}")  # ✅ True
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"transformers:   {transformers.__version__}")   # ✅ 4.36.2
print(f"tokenizers:     {tokenizers.__version__}")     # ✅ 0.15.2
print(f"peft:           {peft.__version__}")            # ✅ 0.7.1
print(f"bitsandbytes:   {bitsandbytes.__version__}")    # ✅ 0.41.1
print(f"accelerate:     {accelerate.__version__}")      # ✅ 0.25.0
print("=" * 50)


# ============================================================
# SECTION 2 — Load PDFs → Text (Cached)
# ============================================================
from pypdf import PdfReader
import os

CACHE_FILE  = "data/extracted_text.txt"
FOLDER_PATH = "data"

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t + "\n"
    return text

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"\n✅ Loaded from cache: {CACHE_FILE}")
    print(f"   Total characters : {len(text):,}")
else:
    if not os.path.exists(FOLDER_PATH):
        raise FileNotFoundError(f"❌ Folder not found: {FOLDER_PATH}")

    text = ""
    pdf_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith(".pdf")]
    print(f"\nFound {len(pdf_files)} PDF(s): {pdf_files}")

    for file in pdf_files:
        extracted = extract_text_from_pdf(os.path.join(FOLDER_PATH, file))
        text += extracted
        print(f"  ✅ {file} → {len(extracted):,} chars")

    os.makedirs(FOLDER_PATH, exist_ok=True)
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"\n💾 Saved to cache  : {CACHE_FILE}")
    print(f"   Total characters: {len(text):,}")

print("\n--- Preview (first 500 chars) ---")
print(text[:500])


# ============================================================
# SECTION 3 — Split Text + Format as Q&A
# ============================================================
from datasets import Dataset

CHUNK_SIZE = 300   # characters per chunk
MIN_CHUNK  = 100   # skip chunks shorter than this

def split_text(text, chunk_size=CHUNK_SIZE):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        if len(chunk.strip()) > MIN_CHUNK:
            chunks.append(chunk)
    return chunks

def format_as_qa(chunks):
    """Wrap each chunk as an instruction-style Q&A prompt."""
    return [
        f"Question: Explain the following law clearly.\nAnswer: {chunk}"
        for chunk in chunks
    ]

chunks           = split_text(text)
formatted_chunks = format_as_qa(chunks)
dataset          = Dataset.from_dict({"text": formatted_chunks})

print(f"\nTotal chunks : {len(chunks)}")
print(f"Dataset size : {len(dataset)}")
print("\nSample entry :")
print(formatted_chunks[0][:400])


# ============================================================
# SECTION 4 — Load Tokenizer
# ============================================================
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

print(f"\n✅ Tokenizer loaded : {MODEL_NAME}")
print(f"   Vocab size       : {tokenizer.vocab_size:,}")


# ============================================================
# SECTION 5 — Tokenize Dataset
# ============================================================
MAX_LENGTH = 256  # keep low for 4 GB VRAM

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

# Step 1: Tokenize
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("\nStep 1 ✅ Tokenized")

# Step 2: Add labels (same as input_ids for causal LM)
tokenized_dataset = tokenized_dataset.map(
    lambda x: {
        "input_ids":      x["input_ids"],
        "attention_mask": x["attention_mask"],
        "labels":         x["input_ids"]   # causal LM: predict next token
    },
    remove_columns=tokenized_dataset.column_names
)
print("Step 2 ✅ Labels added")
print(f"\nFinal dataset : {len(tokenized_dataset)} samples")
print(f"Columns       : {tokenized_dataset.column_names}")


# ============================================================
# SECTION 6 — Load Model in 4-bit (QLoRA)
# ============================================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # saves extra VRAM
    bnb_4bit_quant_type="nf4"         # best for LLMs
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto"                 # auto GPU placement
)

print("\n✅ Model loaded in 4-bit")
print(f"   GPU memory used : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   GPU memory total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ============================================================
# SECTION 7 — Prepare for k-bit Training + Apply LoRA
# ============================================================
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Step 1: Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)
print("\nStep 1 ✅ Model prepared for k-bit training")

# Step 2: Define LoRA config
lora_config = LoraConfig(
    r=8,                                   # rank — higher = more params
    lora_alpha=16,                         # scaling factor
    target_modules=["q_proj", "v_proj"],   # phi-2 attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Step 3: Apply LoRA
model = get_peft_model(model, lora_config)
print("Step 2 ✅ LoRA applied")
print()
model.print_trainable_parameters()         # should show ~1-2% trainable


# ============================================================
# SECTION 8 — Train
# ============================================================
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qlora-law",
    per_device_train_batch_size=1,         # keep 1 for 4 GB VRAM
    gradient_accumulation_steps=8,         # effective batch size = 8
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,                             # mixed precision — saves VRAM
    optim="paged_adamw_8bit",              # memory-efficient optimizer
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",                      # disable wandb
    dataloader_pin_memory=False            # avoid OOM on low VRAM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("\n🚀 Starting training...")
print(f"   Samples  : {len(tokenized_dataset)}")
print(f"   Epochs   : {training_args.num_train_epochs}")
print(f"   Batch    : {training_args.per_device_train_batch_size}")
print(f"   Grad acc : {training_args.gradient_accumulation_steps}")
print()

trainer.train()


# ============================================================
# SECTION 9 — Save Model
# ============================================================
SAVE_DIR = "qlora-law-model"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\n✅ Model saved to : {SAVE_DIR}/")
print(f"   Files          : {os.listdir(SAVE_DIR)}")


# ============================================================
# SECTION 10 — Inference (Test Your Model)
# ============================================================
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16
)

prompts = [
    "You are a legal expert.\nQuestion: What is the Indian Penal Code?\nAnswer:",
    "You are a legal expert.\nQuestion: What is bail?\nAnswer:",
    "You are a legal expert.\nQuestion: What is the Bharatiya Nyaya Sanhita?\nAnswer:",
]

for i, prompt in enumerate(prompts):
    print(f"\n{'=' * 50}")
    print(f"Test {i + 1}:")
    print("=" * 50)
    output = pipe(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1       # avoid repetitive output
    )
    print(output[0]["generated_text"])


# ============================================================
# SECTION 11 — Reload Saved Model (run this instead of
#              sections 6-7 if model is already trained)
# ============================================================
# Uncomment the block below to reload a previously saved model:

# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# SAVE_DIR   = "qlora-law-model"
# MODEL_NAME = "microsoft/phi-2"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     trust_remote_code=True,
#     device_map="auto"
# )
#
# model     = PeftModel.from_pretrained(base_model, SAVE_DIR)
# tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
#
# print(f"✅ Loaded fine-tuned model from: {SAVE_DIR}/")


===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {WindowsPath('/Users/yedee/anaconda3/envs/conda_genai/lib'), WindowsPath('C')}
The following directories listed in your path were found to be non-existent: {WindowsPath('vs/workbench/api/node/extensionHostProcess')}
The following directories listed in your path were found to be non-existent: {WindowsPath('module'), WindowsPath('/matplotlib_inline.backend_inline')}
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
The following directories listed in your path were found to be non-existent: {WindowsPath('/usr/local/cuda/lib64')}
DEBUG: Possible options found for libcudart.so: set()
CUDA SETUP: PyTorch settings found: CUDA_VERSION=118, Highest Compute Capability: 7.5.
CUDA SETUP: To manually override the PyTorch CUDA version please see:https://github.com/TimDettmers/bitsandbytes/blob

c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\bitsandbytes\cuda_setup\main.py:166: UserWarning: Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes


  warn(msg)
c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\bitsandbytes\cuda_setup\main.py:166: UserWarning: C:\Users\yedee\anaconda3\envs\conda_genai did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)


RuntimeError: 
        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues

### 9. Updated code

In [2]:
# ================================
# 1. Install (run once)
# ================================
# pip install torch transformers datasets peft accelerate bitsandbytes pypdf


# ================================
# 2. Load PDFs → Text
# ================================
from pypdf import PdfReader
import os

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t + "\n"
    return text

# ── Cache file path ──────────────────────────────────────────
CACHE_FILE = "data/extracted_text.txt"

# ── Load from cache if it exists, else extract from PDFs ─────
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"✅ Loaded from cache: {CACHE_FILE} ({len(text)} characters)")

else:
    folder_path = "data"
    text = ""

    if not os.path.exists(folder_path):
        print(f" Folder '{folder_path}' not found!")
    else:
        pdf_files = [f for f in os.listdir(folder_path) if f.endswith(".pdf")]
        print(f"Found {len(pdf_files)} PDF(s): {pdf_files}")

        for file in pdf_files:
            pdf_path = os.path.join(folder_path, file)
            extracted = extract_text_from_pdf(pdf_path)
            text += extracted
            print(f" {file} → {len(extracted)} characters")

        # Save to cache
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"\n💾 Saved to cache: {CACHE_FILE}")

print(f"Total characters: {len(text)}")
print("\n--- Preview ---")
print(text[:1000])



✅ Loaded from cache: data/extracted_text.txt (4297307 characters)
Total characters: 4297307

--- Preview ---
1 
 
 
 
 
 
 
 
 
 
 
 
The Bharatiya Nyaya Sanhita, 2023 
 (ACT NO. 45 OF 2023) 
[As on the 6th October, 2025] 
  

2 
 
LIST OF ABBREVIATIONS USED 
G.S.R.              . . . .            .            for           General Statutory Rules.  
S.O.                 . . . . .            ,,              Statutory Order. 
Notifn.  . . . . .  ,,    Notification. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
3 
 
THE BHARATIYA NYAYA SANHITA, 2023 
________ 
ARRANGEMENT OF SECTIONS 
________ 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1. Short title, commencement and application. 
2. Definitions. 
3. General explanations. 
CHAPTER II 
OF PUNISHMENTS 
4. Punishments. 
5. Commutation of sentence. 
6. Fractions of terms of punishment. 
7. Sentence may be (in certain cases of imprisonment) wholly or partly rigorous or simple. 
8. Amount of fine, liability in default of paymen

In [3]:

# ================================
# 3. Split + Format (IMPORTANT)
# ================================
from datasets import Dataset

def split_text(text, chunk_size=300):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        if len(chunk.strip()) > 100:
            chunks.append(chunk)
    return chunks

chunks = split_text(text)

def format_data(chunks):
    formatted = []
    for chunk in chunks:
        formatted.append(
            f"Question: Explain the following law clearly.\nAnswer: {chunk}"
        )
    return formatted

formatted_chunks = format_data(chunks)

dataset = Dataset.from_dict({"text": formatted_chunks})


c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:


# ================================
# 4. Tokenizer
# ================================
from transformers import AutoTokenizer

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\yedee\anaconda3\envs\conda_genai\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\yedee\anaconda3\envs\conda_genai\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\traitlets\config\application.py", line 1075, in launch_in

In [5]:

# ================================
# 5. Tokenization
# ================================
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.map(
    lambda x: {
        "input_ids": x["input_ids"],
        "attention_mask": x["attention_mask"],
        "labels": x["input_ids"]
    },
    remove_columns=tokenized_dataset.column_names
)


Map: 100%|██████████| 14318/14318 [00:19<00:00, 724.84 examples/s]


In [6]:


# ================================
# 6. Load Model (QLoRA - 4bit)
# ================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

# IMPORTANT
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)


False

===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {WindowsPath('/Users/yedee/anaconda3/envs/conda_genai/lib'), WindowsPath('C')}
The following directories listed in your path were found to be non-existent: {WindowsPath('vs/workbench/api/node/extensionHostProcess')}
The following directories listed in your path were found to be non-existent: {WindowsPath('module'), WindowsPath('/matplotlib_inline.backend_inline')}
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
The following directories listed in your path were found to be non-existent: {WindowsPath('/usr/local/cuda/lib64')}
DEBUG: Possible options found for libcudart.so: set()
CUDA SETUP: PyTorch settings found: CUDA_VERSION=118, Highest Compute Capability: 7.5.
CUDA SETUP: To manually override the PyTorch CUDA version please see:https://github.com/TimDettmers/bitsandbyte

c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\bitsandbytes\cuda_setup\main.py:166: UserWarning: Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes


  warn(msg)
c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\bitsandbytes\cuda_setup\main.py:166: UserWarning: C:\Users\yedee\anaconda3\envs\conda_genai did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)


RuntimeError: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):

        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues

In [12]:
!pip uninstall torch torchaudio torchvision -y

Found existing installation: torch 2.1.0+cu118
Uninstalling torch-2.1.0+cu118:
  Successfully uninstalled torch-2.1.0+cu118
Found existing installation: torchaudio 2.1.0+cu118
Uninstalling torchaudio-2.1.0+cu118:
  Successfully uninstalled torchaudio-2.1.0+cu118
Found existing installation: torchvision 0.16.0+cu118
Uninstalling torchvision-0.16.0+cu118:
  Successfully uninstalled torchvision-0.16.0+cu118


In [13]:
!pip install torch==2.1.0+cu118 torchaudio==2.1.0+cu118 torchvision==0.16.0+cu118 --index-url https://download.pytorch.org/whl/cu118


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.1.0%2Bcu118-cp310-cp310-win_amd64.whl (2722.7 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.1.0%2Bcu118-cp310-cp310-win_amd64.whl (3.9 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.16.0%2Bcu118-cp310-cp310-win_amd64.whl (5.0 MB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   -----------------------------

In [1]:
pip list

Package                                  Version
---------------------------------------- ------------
accelerate                               0.25.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
altair                                   6.0.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anyio                                    4.13.0
asttokens                                3.0.1
async-timeout                            4.0.3
attrs                                    26.1.0
bcrypt                                   5.0.0
beautifulsoup4                           4.14.3
bitsandbytes                             0.41.1
blinker                                  1.9.0
build                                    1.4.0
cachetools                               7.0.5
certifi                                  2026.2.25
cffi                                     

In [6]:
!pip install \
  "torch==2.1.0+cu118" \
  "transformers==4.36.2" \
  "tokenizers==0.15.2" \
  "bitsandbytes==0.41.1" \
  "peft==0.7.1" \
  "accelerate==0.25.0" \
  "huggingface_hub==0.20.3" \
  --index-url https://download.pytorch.org/whl/cu118 \
  --extra-index-url https://pypi.org/simple \
  --no-deps \
  --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu118, https://pypi.org/simple
  Using cached https://download.pytorch.org/whl/cu118/torch-2.1.0%2Bcu118-cp310-cp310-win_amd64.whl (2722.7 MB)
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached tokenizers-0.15.2-cp310-none-win_amd64.whl.metadata (6.8 kB)
  Using cached bitsandbytes-0.41.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached peft-0.7.1-py3-none-any.whl.metadata (25 kB)
  Using cached accelerate-0.25.0-py3-none-any.whl.metadata (18 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
Using cached tokenizers-0.15.2-cp310-none-win_amd64.whl (2.2 MB)
Using cached bitsandbytes-0.41.1-py3-none-any.whl (92.6 MB)
Using cached peft-0.7.1-py3-none-any.whl (168 kB)
Using cached accelerate-0.25.0-py3-none-any.whl (265 kB)

  Attempting uninstall: bitsandbytes

    Found existing installation: bitsandbytes 0.41.1

   ---------------------------------------- 0/7 [bitsandbytes]
    Uninstalli

  You can safely remove it manually.


In [ ]:


# ================================
# 7. Apply LoRA
# ================================
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)



In [ ]:

# ================================
# 8. Training
# ================================
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qlora-law",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


In [ ]:


# ================================
# 9. Save Model
# ================================
model.save_pretrained("qlora-law-model")
tokenizer.save_pretrained("qlora-law-model")


In [ ]:


# ================================
# 10. Inference
# ================================
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = "You are a legal expert.\nQuestion: What is Indian Penal Code?\nAnswer:"

output = pipe(prompt, max_length=200)

print(output[0]["generated_text"])

In [4]:
# ================================
# 1. Install (run once)
# ================================
# pip install torch transformers datasets peft accelerate bitsandbytes pypdf


# ================================
# 2. Load PDFs → Text
# ================================
from pypdf import PdfReader
import os

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t
    return text

text = ""

folder_path = "data"

for file in os.listdir(folder_path):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(folder_path, file)
        text += extract_text_from_pdf(pdf_path)

print(text[:1000])


# ================================
# 3. Split + Format (IMPORTANT)
# ================================
from datasets import Dataset

def split_text(text, chunk_size=300):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        if len(chunk.strip()) > 100:
            chunks.append(chunk)
    return chunks

chunks = split_text(text)

def format_data(chunks):
    formatted = []
    for chunk in chunks:
        formatted.append(
            f"Question: Explain the following law clearly.\nAnswer: {chunk}"
        )
    return formatted

formatted_chunks = format_data(chunks)

dataset = Dataset.from_dict({"text": formatted_chunks})


# ================================
# 4. Tokenizer
# ================================
from transformers import AutoTokenizer

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# ================================
# 5. Tokenization
# ================================
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.map(
    lambda x: {
        "input_ids": x["input_ids"],
        "attention_mask": x["attention_mask"],
        "labels": x["input_ids"]
    },
    remove_columns=tokenized_dataset.column_names
)


# ================================
# 6. Load Model (QLoRA - 4bit)
# ================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

# IMPORTANT
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)


# ================================
# 7. Apply LoRA
# ================================
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


# ================================
# 8. Training
# ================================
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qlora-law",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


# ================================
# 9. Save Model
# ================================
model.save_pretrained("qlora-law-model")
tokenizer.save_pretrained("qlora-law-model")


# ================================
# 10. Inference
# ================================
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = "You are a legal expert.\nQuestion: What is Indian Penal Code?\nAnswer:"

output = pipe(prompt, max_length=200)

print(output[0]["generated_text"])

1 
 
 
 
 
 
 
 
 
 
 
 
The Bharatiya Nyaya Sanhita, 2023 
 (ACT NO. 45 OF 2023) 
[As on the 6th October, 2025] 
  
2 
 
LIST OF ABBREVIATIONS USED 
G.S.R.              . . . .            .            for           General Statutory Rules.  
S.O.                 . . . . .            ,,              Statutory Order. 
Notifn.  . . . . .  ,,    Notification. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
THE BHARATIYA NYAYA SANHITA, 2023 
________ 
ARRANGEMENT OF SECTIONS 
________ 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1. Short title, commencement and application. 
2. Definitions. 
3. General explanations. 
CHAPTER II 
OF PUNISHMENTS 
4. Punishments. 
5. Commutation of sentence. 
6. Fractions of terms of punishment. 
7. Sentence may be (in certain cases of imprisonment) wholly or partly rigorous or simple. 
8. Amount of fine, liability in default of payment of fine, etc. 
9. Limit of punishment of offence made up of several offences. 
10. Punishment of person guilt

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Map: 100%|██████████| 14314/14314 [00:21<00:00, 677.20 examples/s]
`low_cpu_mem_usage` was None, now set to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 2/2 [00:26<00:00, 13.11s/it]


ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [3]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA GeForce GTX 1650 Ti
VRAM: 4.3 GB
